In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


In [2]:
#Φόρτωση δεδομένων από το OpenML.

datasets_list = ['breast-w', 'diabetes', 'credit-g']

# Αλγόριθμοι προς εκπαίδευση
classifiers = {
    'k-NN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(random_state=42),
    'MLP (Neural Net)': MLPClassifier(max_iter=500, random_state=42)
}

results_data = []


In [3]:

for ds_name in datasets_list:



        dataset = fetch_openml(name=ds_name, as_frame=True, parser='auto')
    except Exception as e:
        print(f"Σφάλμα στο {ds_name}: {e}")
        continue

    X = dataset.data
    y = dataset.target

    print(f"\nDATASET: {ds_name}")

    # Αριθμός δειγμάτων και χαρακτηριστικών
    print(f"Δείγματα: {X.shape[0]}, Χαρακτηριστικά: {X.shape[1]}")

    # Τύποι χαρακτηριστικών (αριθμητικά / κατηγορικά)
    print(f"Τύποι χαρακτηριστικών:\n{X.dtypes.value_counts()}")

    #  Ύπαρξη ελλιπών τιμών
    print(f"Ελλιπείς τιμές: {X.isnull().sum().sum()}")

    # Κωδικοποίηση στόχου
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        y = le.fit_transform(y)

    #  Προεπεξεργασία


    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X.select_dtypes(include=['object', 'category']).columns

    #  Χειρισμός ελλιπών (SimpleImputer) & Κανονικοποίηση (StandardScaler)
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])

    #  Κωδικοποίηση κατηγορικών (OneHotEncoder)
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    # Χρήση Pipeline και ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])


    # Εκπαίδευση & Αξιολόγηση

    for clf_name, clf in classifiers.items():
        model_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', clf)
        ])

        #  Επιλογή μετρικών: accuracy, precision, recall, F1
        scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

        #  Αξιολόγηση με cross-validation
        cv_results = cross_validate(model_pipeline, X, y, cv=5, scoring=scoring)

        results_data.append({
            'Dataset': ds_name,
            'Algorithm': clf_name,
            'Accuracy': np.mean(cv_results['test_accuracy']),
            'Precision': np.mean(cv_results['test_precision_macro']),
            'Recall': np.mean(cv_results['test_recall_macro']),
            'F1-Score': np.mean(cv_results['test_f1_macro'])
        })


DATASET: breast-w
Δείγματα: 699, Χαρακτηριστικά: 9
Τύποι χαρακτηριστικών:
int64      8
float64    1
Name: count, dtype: int64
Ελλιπείς τιμές: 16


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/datasets/_openml.py:323: UserWarning: Multiple active versions of the dataset matching the name diabetes exist. Versions may be fundamentally different, returning version 1. Available versions:
- version 1, status: active
  url: https://www.openml.org/search?type=data&id=37
- version 5, status: active
  url: https://www.openml.org/search?type=data&id=42608

  warn(warning_msg)



DATASET: diabetes
Δείγματα: 768, Χαρακτηριστικά: 8
Τύποι χαρακτηριστικών:
int64      6
float64    2
Name: count, dtype: int64
Ελλιπείς τιμές: 0


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/datasets/_openml.py:323: UserWarning: Multiple active versions of the dataset matching the name credit-g exist. Versions may be fundamentally different, returning version 1. Available versions:
- version 1, status: active
  url: https://www.openml.org/search?type=data&id=31
- version 2, status: active
  url: https://www.openml.org/search?type=data&id=44096

  warn(warning_msg)



DATASET: credit-g
Δείγματα: 1000, Χαρακτηριστικά: 20
Τύποι χαρακτηριστικών:
int64       7
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
Name: count, dtype: int64
Ελλιπείς τιμές: 0


In [5]:
#αποτελεσμάτα σε συγκριτικό πίνακα
results_df = pd.DataFrame(results_data)

results_df = results_df.sort_values(by=['Dataset', 'Accuracy'], ascending=[True, False])

results_df_styled = results_df.round(4)

print("\n--- Συγκριτικός Πίνακας Αποτελεσμάτων ---")
display(results_df_styled)


--- Συγκριτικός Πίνακας Αποτελεσμάτων ---


,Dataset,Algorithm,Accuracy,Precision,Recall,F1-Score
1,breast-w,Random Forest,0.9671,0.9645,0.9641,0.9637
0,breast-w,k-NN,0.9643,0.9623,0.9600,0.9604
2,breast-w,MLP (Neural Net),0.9643,0.9599,0.9619,0.9604
7,credit-g,Random Forest,0.7540,0.7197,0.6490,0.6604
8,credit-g,MLP (Neural Net),0.7330,0.6783,0.6645,0.6699
6,credit-g,k-NN,0.7200,0.6564,0.6067,0.6120
5,diabetes,MLP (Neural Net),0.7644,0.7436,0.7299,0.7349
4,diabetes,Random Forest,0.7631,0.7449,0.7185,0.7266
3,diabetes,k-NN,0.7357,0.7093,0.6923,0.6980
